# Deposit Attrition EDA — v8

**Payment Knowledge Graph (PKG) · PNC Treasury Management · Data Science**

v7 established the calendar-time frame, closed §4.2, and produced a queue that beats the incumbent
on every axis at matched volume. It also surfaced the thing this notebook is built around:

> `fin_out_n` — the signal this programme has briefed for months — is derived from
> `cpty_fin_entity_name` **in the outbound direction, populated 37.4% of the time**. It is the
> sparsest field in the payment table. The same field is populated **86.3% inbound**, and
> `cpty_name` is populated **99.9% outbound**. Neither has been built.

The earlier reading — that coverage was blocked on the `PAYS_CPTY` / `CptyFinEntity` ingestion —
was wrong. **Those milestones belong to the Neo4j graph. This analysis does not use the graph.**
It reads `dsihd01p_dsi.neo4j_payments` directly: 12.8 billion transactions carrying a date, a rail,
an amount, a counterparty name, a counterparty account and the counterparty's bank, on both
directions. Everything in v8 is built from data already queryable today.

## The question that gates everything else

**Is the institution field populated as a function of payment rail?** A wire carries a beneficiary
bank routing number; a cheque does not; an ACH origination may or may not resolve the receiving
institution. If `cpty_fin_entity_name` is largely a wire artefact, then `fin_out_n` is a
*wire-activity signal in disguise*, its 37% coverage is the wire share of outbound volume, and
"banks they pay drop away" means something much narrower than we have been saying.

§4 answers this two ways: structurally, from the source table, and behaviourally, from data already
on disk — the second is decisive and costs nothing.

**Read §4 before letting §6 run.** If the field is rail-determined, the counterparty rebuild
changes shape.

## Blocks

| § | What | Cost |
|---|---|---|
| 2 | **Inventory.** Every artefact previous versions wrote, and every column in both source tables with its population rate — including the ones we have never read | Minutes |
| 3 | Field coverage on the payment table, by direction, confirming `02 §1.1` on current data | ~10 min |
| 4 | **Rail × institution.** The gating question, structurally and behaviourally | ~15 min |
| 5 | **Counterparty key QA.** Name↔account cardinality, generic-name registry, under- and over-merge measured before we switch keys | ~20 min |
| 6 | Per-month enriched build — name-keyed pairs, rail detail, self-payment. Resumable, one job per month | Hours |
| 7 | New feature families: inbound counterparties and institutions, recurring-series breakage, timing, self-payment to a competitor | ~30 min |
| 8 | Cheap features off panels already on disk: trend, run-length, rail-mix shift, concentration, account trajectory | ~10 min |
| 9 | **Ablation** on the v7 harness — every block measured separately, same folds, same clients | ~20 min |

## Deliberately out of scope

Customer attributes — segment, NAICS, size band, product holdings. Those arrive after this
notebook establishes what the payment and deposit data alone can do, so the lift they add is
measurable against a fixed baseline rather than confounded with a feature rebuild.


## 0 · Configuration

In [ ]:
# =====================================================================
# 0 · CONFIGURATION — v8
# =====================================================================
from pathlib import Path

PAY_TBL = "dsihd01p_dsi.neo4j_payments"
DEP_TBL = "dsihd01p_dsi.lap_dsi_universe_optimized"

HDFS_V2  = "hdfs://nameservice1/user/pk36814/attrition_v2"
HDFS_V6  = "hdfs://nameservice1/user/pk36814/attrition_v6"
HDFS_V7  = "hdfs://nameservice1/user/pk36814/attrition_v7"
HDFS_DIR = "hdfs://nameservice1/user/pk36814/attrition_v8"
OUT_DIR  = Path("/projects/DSI/sa15474/repos/pkg/eda/attrition_v8")
# TRAP (02 §6): pathlib collapses hdfs://host/p -> hdfs:/host/p. Local Path
# and HDFS string stay separate variables and are never mixed.

DATE_START, DATE_END = "2024-01-01", "2026-07-31"
MAX_ROWS, SEED = 60, 20260907

# ── stage switches. §2-§5 are the audit and are cheap. §6 is hours. ───
RUN_INVENTORY = True
RUN_COVERAGE  = True     # §3 — one pass per sample month over the payment table
RUN_RAIL      = True     # §4 — THE GATING QUESTION
RUN_KEY_QA    = True     # §5 — before switching counterparty keys
RUN_BUILD     = True     # §6 — per-month, resumable, skip-if-exists
RUN_FEATURES  = True
RUN_ABLATION  = True

AUDIT_MONTHS  = ["2024-06", "2025-06", "2026-06"]   # §3/§4 structural sample:
                                                    # one per year, mid-year, to
                                                    # separate a stable property
                                                    # from a drifting one

# ── carried from v7 unchanged so dd is comparable ─────────────────────
CHG_LAG_FAR, CHG_LAG_NEAR = -6, -4
CHG_MIN_OBS, MIN_REF      = 2, 1.0
PEER_DECILES, PEER_ANCHOR_M, PEER_MIN_N = 10, 3, 50
PEER_USE_SEG = False          # segment_desc is restated; leakage
DD_WARMUP_M  = 6              # OFFSET. m_idx is ABSOLUTE (~24289-24319);
                              # resolved against M_MIN in §7, never hard-coded
ORIGIN_START_OFF, MAX_ORIGINS = 18, 24
HORIZONS, PRIMARY_H = [1, 3, 6], 6
PRIMARY_DEF = "A_full_exit"
NEG_SAMPLE, TEST_STAYER_FRAC = 0.15, 1.0
L2, IRLS_MAX_IT, IRLS_TOL = 2.0, 60, 1e-9
DD_CLIP = (0.01, 100.0)
CAPACITY, QUEUE_K = [50, 100, 250, 500, 1000, 2500], 250
MAX_COLLECT_ROWS = 3_000_000
MIN_TRAIN_POS = 300

# ── §5 counterparty key normalisation ─────────────────────────────────
LEGAL_SUFFIX = ["INC", "INCORPORATED", "LLC", "L L C", "LTD", "LIMITED", "CORP",
                "CORPORATION", "CO", "COMPANY", "LP", "LLP", "PLC", "PC", "PA",
                "TRUST", "THE"]
# A normalised name paid by more than this many DISTINCT customers is generic
# ("PAYROLL", "ACH CREDIT", "TRANSFER") or a processor. Median-multiple, not a
# percentile — a percentile flags a fixed fraction by construction, which is the
# mistake the locatability work documented.
GENERIC_MEDIAN_MULT = 15
GENERIC_MIN_PAYERS  = 500      # floor, so the rule cannot fire on a thin month

# ── §6/§7 build ───────────────────────────────────────────────────────
PAIR_TOP_N   = 200      # counterparties kept per customer-month, by amount.
                        # The tail past 200 is one-off payments and it is the
                        # join cost, not the signal
REC_WINDOW   = 6        # months of history defining a recurring series
REC_MIN_HITS = 4        # present in >= this many of REC_WINDOW = "standing"
SELF_SIM_MIN = 0.90     # Jaro-Winkler floor for "counterparty is the customer"
NON_PNC_ONLY = True     # self-payment only counts when the destination bank
                        # is not PNC — that is the whole point

FEATURE_BLOCKS = ["v7_base", "trend", "railmix", "concentration", "accounts",
                  "cpty_name_out", "cpty_in", "fin_in", "recurring", "timing",
                  "selfpay"]
HTML_NAME = "PKG_Attrition_v8_Features.html"


In [ ]:
# =====================================================================
# 1 · IMPORTS, SESSION, HELPERS
# =====================================================================
import warnings, time, math, json, datetime as dt
import numpy as np, pandas as pd
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark import StorageLevel
from IPython.display import display, HTML

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=ResourceWarning)
spark = (SparkSession.builder.appName("pkg_attrition_eda_v8")
         .config("spark.sql.shuffle.partitions", "800")   # v8 shuffles the pair
                                                          # panel; 400 spills
         .config("spark.sql.execution.arrow.pyspark.enabled", "false")
         # KEEP OFF. PySpark 3.3.2's arrow path references np.object0 / np.bool8,
         # both removed in numpy 2.0. Not only the decimal(15,0) case.
         .config("spark.sql.autoBroadcastJoinThreshold", str(64*1024*1024))
         .enableHiveSupport().getOrCreate())
pd.set_option("display.max_columns", 400); pd.set_option("display.width", 260)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"numpy {np.__version__} · pandas {pd.__version__} · spark {spark.version}")

def hp(n): return f"{HDFS_DIR.rstrip('/')}/{n}"
def v2(n): return f"{HDFS_V2.rstrip('/')}/{n}"
def v6(n): return f"{HDFS_V6.rstrip('/')}/{n}"
def v7(n): return f"{HDFS_V7.rstrip('/')}/{n}"
def pct(a, b): return float(a)/float(b) if b else float("nan")

def _dec(s):
    o = s
    for c, t in s.dtypes:
        if t.startswith("decimal"): o = o.withColumn(c, F.col(c).cast("double"))
    return o

def exists(path):
    try:
        spark.read.parquet(path).limit(1).count(); return True
    except Exception:
        return False

def read_first(paths, label):
    for p in paths:
        if exists(p):
            print(f"  {label}: reusing {p}"); return spark.read.parquet(p)
    print(f"  {label}: absent from {len(paths)} location(s)"); return None

def collect_pd(sdf, label="", max_rows=None):
    max_rows = MAX_COLLECT_ROWS if max_rows is None else max_rows
    n = sdf.count()
    if n > max_rows:
        raise RuntimeError(f"{label}: {n:,} rows > MAX_COLLECT_ROWS={max_rows:,}")
    t0 = time.time(); out = _dec(sdf).toPandas()
    print(f"  collected {label}: {n:,} x {out.shape[1]} in {time.time()-t0:,.0f}s")
    return out

def disp(obj, title=None, n=None, save=None, transpose=False):
    n = MAX_ROWS if n is None else n
    out = _dec(obj).limit(n).toPandas() if hasattr(obj, "toPandas") else (
        obj.copy() if isinstance(obj, pd.DataFrame) else pd.DataFrame(obj))
    if save: out.to_csv(OUT_DIR / f"{save}.csv", index=False)
    if title:
        display(HTML(f"<div style='font:600 13px/1.6 IBM Plex Sans,sans-serif;"
                     f"margin:10px 0 2px;color:#111'>{title}"
                     f"<span style='font-weight:400;color:#888'> &middot; {len(out)} rows"
                     f"</span></div>"))
    display(out.T if transpose else out)
    return out

def kv(pairs, title=None, save=None):
    """Takes ORDERED PAIRS and raises on a repeated label — the v6 §5h bug
    (duplicate dict keys silently printing only the last value) cannot recur."""
    items = list(pairs.items()) if isinstance(pairs, dict) else list(pairs)
    labs = [k for k, _ in items]
    dup = sorted({k for k in labs if labs.count(k) > 1})
    if dup: raise ValueError(f"kv(): duplicate labels {dup}")
    return disp(pd.DataFrame({"metric": labs, "value": [v for _, v in items]}),
                title=title, n=len(items), save=save)

def auc(y, s):
    """Rank-based Mann-Whitney. Non-finite scores DROPPED, not imputed — an
    unscoreable client is not a coin flip. No np.trapz (gone in numpy 2)."""
    y = np.asarray(y, float); s = np.asarray(s, float)
    ok = np.isfinite(s) & np.isfinite(y); y, s = y[ok], s[ok]
    n1 = float(y.sum()); n0 = float(len(y) - n1)
    if n1 == 0 or n0 == 0: return np.nan
    r = pd.Series(s).rank(method="average").to_numpy()
    return float((r[y == 1].sum() - n1*(n1+1)/2.0)/(n1*n0))

def topk_metrics(score, y, ks, p_base=None):
    s = np.asarray(score, float); y = np.asarray(y, float)
    n_all = len(s); fin = np.isfinite(s)
    s = np.where(fin, s, -np.inf)
    o = np.argsort(-s, kind="stable"); ys = y[o]
    tot = float(y.sum()); p_base = (tot/n_all) if p_base is None else p_base
    rows = []
    for K in ks:
        k = int(min(K, n_all)); tp = float(ys[:k].sum()); prec = tp/k if k else np.nan
        rows.append(dict(k=K, n_at_risk=n_all, queue_coverage=fin.mean(), tp=int(tp),
                         precision=prec, recall=(tp/tot if tot else np.nan),
                         lift=(prec/p_base if p_base > 0 else np.nan)))
    return pd.DataFrame(rows)

def logit_irls(X, y, l2=L2, max_iter=IRLS_MAX_IT, tol=IRLS_TOL):
    X = np.asarray(X, np.float64); y = np.asarray(y, np.float64)
    b = np.zeros(X.shape[1]); R = l2*np.eye(X.shape[1]); R[0, 0] = 0.0
    for _ in range(max_iter):
        eta = np.clip(X @ b, -30, 30); mu = 1/(1+np.exp(-eta))
        w = np.maximum(mu*(1-mu), 1e-6); z = eta + (y-mu)/w
        XtW = X.T * w
        try: bn = np.linalg.solve(XtW @ X + R, XtW @ z)
        except np.linalg.LinAlgError:
            bn = np.linalg.lstsq(XtW @ X + R, XtW @ z, rcond=None)[0]
        if np.max(np.abs(bn-b)) < tol: b = bn; break
        b = bn
    return b

def fit_spec(tr, cols, s_neg=NEG_SAMPLE):
    """Standardise, fit, un-standardise. Zero-variance columns dropped here —
    several dd columns are null by design and arrive constant."""
    X = tr[cols].to_numpy(np.float64); y = tr["y"].to_numpy(np.float64)
    keep = X.std(axis=0) > 1e-9
    cols_k = [c for c, k in zip(cols, keep) if k]
    if not cols_k: return None
    Xk = X[:, keep]; mu = Xk.mean(0); sd = Xk.std(0)
    b = logit_irls(np.column_stack([np.ones(len(Xk)), (Xk-mu)/sd]), y)
    return dict(cols=cols_k, beta=b[1:]/sd,
                b0=float(b[0]-float(np.sum(b[1:]*mu/sd))) + math.log(s_neg),
                n=len(tr), n_pos=int(y.sum()))

def apply_spec(sp, df):
    if sp is None: return np.full(len(df), np.nan)
    return sp["b0"] + df[sp["cols"]].to_numpy(np.float64) @ sp["beta"]

_F = OUT_DIR / "FINDINGS_v8.csv"
FINDINGS = pd.read_csv(_F).to_dict("records") if _F.exists() else []
def note(qid, q, a, d=""):
    global FINDINGS
    FINDINGS = [f for f in FINDINGS if f["id"] != qid]
    FINDINGS.append(dict(id=qid, question=q, answer=str(a), detail=str(d)))
    pd.DataFrame(FINDINGS).to_csv(_F, index=False)
print("helpers ready")


## 2 · Inventory — what already exists, and what we have never read

Two separate audits. First, every artefact previous versions wrote to HDFS, so nothing gets rebuilt
that already exists. Second, **every column in both source tables with its population rate**,
including the ones no version has ever touched. The second is the one that matters: v7's central
finding was that we built on the sparsest available field, and the only defence against repeating
that is to look at all of them.

In [ ]:
# =====================================================================
# 2 · ARTEFACT INVENTORY                                 [OUTPUT BLOCK 1]
# =====================================================================
ART = [("v2 · account-month panel",     v2("panel_account_month")),
       ("v2 · customer-month panel",    v2("panel_customer_month")),
       ("v2 · payment features (21)",   v2("panel_pay_features")),
       ("v2 · pay_pairs (cpty|fin)",    v2("pay_pairs")),
       ("v6 · frozen peer anchor",      v6("peer_anchor")),
       ("v6 · labels",                  v6("labels_customer")),
       ("v7 · calendar-time risk set",  v7("risk_set")),
       ("v7 · monthly incumbent flag",  v7("p30_month")),
       ("v7 · counterparty payer counts", v7("k_payers")),
       ("v8 · enriched pairs",          hp("pairs")),
       ("v8 · rail detail",             hp("rail")),
       ("v8 · self-payment",            hp("selfpay"))]
rows = []
for lab, p in ART:
    ok = exists(p)
    n = ncol = None
    if ok:
        d = spark.read.parquet(p); ncol = len(d.columns)
        try: n = d.count()
        except Exception: n = None
    rows.append(dict(artefact=lab, present=ok, rows=n, cols=ncol, path=p.split("/")[-1]))
INV = disp(pd.DataFrame(rows), title="2a &middot; What previous versions already wrote. "
           "Anything present is reused, not rebuilt", n=30, save="v8_inventory")

# ── column-level coverage, both source tables ─────────────────────────
def col_coverage(tbl, where, label, sample_frac=None):
    """Population rate for EVERY column, not just the ones we use. Computed in
    one pass: len(cols) aggregates over a bounded slice."""
    d = spark.table(tbl).filter(where)
    if sample_frac: d = d.sample(False, sample_frac, SEED)
    cols = d.columns
    aggs = [F.count(F.lit(1)).alias("_n")]
    for c in cols:
        aggs.append(F.avg(F.when(F.col(f"`{c}`").isNotNull() &
                    (F.trim(F.col(f"`{c}`").cast("string")) != ""), 1.0)
                    .otherwise(0.0)).alias(f"p__{c}"))
    r = d.agg(*aggs).collect()[0].asDict()
    out = pd.DataFrame([dict(table=label, column=c, populated=r[f"p__{c}"]) for c in cols])
    out["rows_sampled"] = r["_n"]
    return out.sort_values("populated")

if RUN_INVENTORY:
    t0 = time.time()
    DEP_COV = col_coverage(DEP_TBL, f"edw_tda_load_dt BETWEEN '{DATE_START}' AND '{DATE_END}'",
                           "deposits")
    DEP_USED = {"acct_full_acct_id","sub_product_cd","edw_tda_load_dt","balance",
                "avg_monthly_bal_1","acct_status","acct_status_desc","deposit_family",
                "sub_product_desc","account_type","opened_dt","closed_dt","cust_pwr_id",
                "cust_name","rltn_pwr_id","data_source","src_system_cd","segment_desc",
                "market_desc","state","lob_indicator","cust_naics_cd_val",
                "bdh_hdfs_load_ts","cod_hdfs_load_ts"}
    DEP_COV["used_today"] = DEP_COV.column.isin(DEP_USED)
    disp(DEP_COV[(~DEP_COV.used_today) & (DEP_COV.populated > 0.5)]
         .sort_values("populated", ascending=False),
         title="2b &middot; Deposit columns we have <b>never read</b> that are populated on more "
               "than half of rows. Fee, overdraft and rate fields would live here",
         n=60, save="v8_dep_unused_columns")
    disp(DEP_COV.sort_values("populated"), title="2c &middot; Every deposit column, by population "
         "rate", n=90, save="v8_dep_coverage")

    PAY_COV = col_coverage(PAY_TBL, f"trans_dt BETWEEN '{AUDIT_MONTHS[1]}-01' AND "
                                    f"'{AUDIT_MONTHS[1]}-28'", "payments")
    disp(PAY_COV, title=f"2d &middot; Every payment column, by population rate "
         f"(sample month {AUDIT_MONTHS[1]}). <b>Read this against which fields our features "
         f"actually use</b>", n=60, save="v8_pay_coverage")
    print(f"  inventory in {time.time()-t0:,.0f}s")


## 3 · Coverage by direction, on current data

`02 §1.1` records `cpty_name` 0.999 out / 0.876 in, `unq_cpty_acct_id` 0.395 / 0.927,
`cpty_fin_entity_name` 0.374 / 0.863. Those figures are the basis of the whole v8 argument, so they
get re-measured on current data rather than quoted. Direction follows the documented rule: both mdm
ids present → `internal_c2c`; only `_receives` → `inbound_cpty`; only `_pays` → `outbound_cpty`.

Three sample months, one per year, so a stable structural property is distinguishable from a
drifting one.

In [ ]:
# =====================================================================
# 3 · FIELD COVERAGE BY DIRECTION                        [OUTPUT BLOCK 2]
# =====================================================================
FIELDS = ["cpty_name", "unq_cpty_acct_id", "cpty_fin_entity_name", "payment_rail", "category"]

def _pop(c):
    return F.when(F.col(c).isNotNull() & (F.trim(F.col(c)) != "") &
                  (~F.upper(F.trim(F.col(c))).isin("NULL", "NA", "N/A", "UNKNOWN", "-1")),
                  1.0).otherwise(0.0)

def month_slice(ym):
    lo = f"{ym}-01"
    hi = (pd.Timestamp(lo) + pd.offsets.MonthEnd(1)).strftime("%Y-%m-%d")
    return (spark.table(PAY_TBL).filter(F.col("trans_dt").between(lo, hi))
            .withColumn("direction",
                F.when(F.col("mdm_id_pays").isNotNull() & F.col("mdm_id_receives").isNotNull(),
                       "internal_c2c")
                 .when(F.col("mdm_id_receives").isNotNull(), "inbound_cpty")
                 .when(F.col("mdm_id_pays").isNotNull(), "outbound_cpty")
                 .otherwise("neither")))

if RUN_COVERAGE:
    t0 = time.time(); parts = []
    for ym in AUDIT_MONTHS:
        d = month_slice(ym)
        agg = [F.count(F.lit(1)).alias("n_txn"), F.sum("trans_amt").alias("amt")]
        agg += [F.avg(_pop(c)).alias(c) for c in FIELDS]
        p = d.groupBy("direction").agg(*agg).toPandas(); p.insert(0, "ym", ym)
        parts.append(p)
        print(f"  {ym} done ({time.time()-t0:,.0f}s)")
    COV3 = pd.concat(parts, ignore_index=True)
    COV3.to_csv(OUT_DIR / "v8_direction_coverage.csv", index=False)
    disp(COV3.sort_values(["direction", "ym"]),
         title="3a &middot; Field population by direction and month. <b>Stable across two years "
               "means this is structure, not a data-quality incident</b>", n=20)
    piv = (COV3[COV3.direction.isin(["outbound_cpty", "inbound_cpty"])]
           .groupby("direction")[FIELDS].mean().T.round(3))
    piv["inbound_advantage"] = (piv.get("inbound_cpty") / piv.get("outbound_cpty")).round(2)
    disp(piv.reset_index().rename(columns={"index": "field"}),
         title="3b &middot; The asymmetry, averaged over the three months. <b>Any field where "
               "inbound_advantage &gt; 1 is a signal we built on the worse half</b>",
         save="v8_direction_asymmetry")
    note("COV", "Do the documented coverage figures hold on current data?",
         "see v8_direction_coverage.csv",
         "cpty_fin_entity_name outbound is the field fin_out_n is built on. If it reproduces near "
         "0.374 while inbound reproduces near 0.863, the v8 premise holds.")


## 4 · Rail × institution — the gating question

Is `cpty_fin_entity_name` populated *because of what the payment is*? A wire carries a beneficiary
bank; a cheque carries an image and a payee line; an ACH origination may or may not resolve the
receiving institution depending on the path. If the field is essentially wire-only, then:

- `fin_out_n` is a **wire counterparty count**, not a bank count;
- its 37% coverage is the wire share of outbound activity, and no amount of rebuilding lifts it;
- "banks they pay drop away" should be restated as "their wire relationships thin out", which is a
  narrower and much less impressive claim;
- and the honest fix is a rail-conditional feature, not a coverage fix.

**§4a–c** answer this structurally from the source table. **§4d–f** answer it behaviourally from
data already on disk — no new pass over 12.8 billion rows — and that half is decisive. If a client
having a `fin_out_n` value is nearly the same event as that client sending a wire, the signal is
the rail.

In [ ]:
# =====================================================================
# 4 · RAIL x INSTITUTION — STRUCTURAL                    [OUTPUT BLOCK 3]
# =====================================================================
if RUN_RAIL:
    t0 = time.time(); parts = []
    for ym in AUDIT_MONTHS:
        d = month_slice(ym).filter(F.col("direction").isin("outbound_cpty", "inbound_cpty"))
        p = (d.groupBy("direction", "payment_rail").agg(
                F.count(F.lit(1)).alias("n_txn"),
                F.sum("trans_amt").alias("amt"),
                F.avg(_pop("cpty_fin_entity_name")).alias("has_fin"),
                F.avg(_pop("unq_cpty_acct_id")).alias("has_acct"),
                F.avg(_pop("cpty_name")).alias("has_name"))).toPandas()
        p.insert(0, "ym", ym); parts.append(p)
        print(f"  {ym} rail x field ({time.time()-t0:,.0f}s)")
    RAIL = (pd.concat(parts, ignore_index=True)
            .groupby(["direction", "payment_rail"], as_index=False)
            .agg(n_txn=("n_txn", "sum"), amt=("amt", "sum"),
                 has_fin=("has_fin", "mean"), has_acct=("has_acct", "mean"),
                 has_name=("has_name", "mean")))
    RAIL["txn_share_of_direction"] = RAIL.n_txn / RAIL.groupby("direction").n_txn.transform("sum")
    RAIL = RAIL.sort_values(["direction", "n_txn"], ascending=[True, False])
    disp(RAIL, title="4a &middot; <b>Population of the institution field by rail.</b> If has_fin "
         "is near 1.0 on WIRE and near 0 on CHECK, the field is a property of the rail",
         n=40, save="v8_rail_field_population")

    # Where do the populated institution rows actually come from?
    OUT = RAIL[RAIL.direction == "outbound_cpty"].copy()
    OUT["fin_txns"] = OUT.n_txn * OUT.has_fin
    OUT["share_of_all_populated"] = OUT.fin_txns / OUT.fin_txns.sum()
    disp(OUT[["payment_rail", "n_txn", "txn_share_of_direction", "has_fin",
              "share_of_all_populated"]].sort_values("share_of_all_populated", ascending=False),
         title="4b &middot; <b>Of every OUTBOUND transaction that carries an institution, which "
               "rail did it come from?</b> This is the composition of fin_out_n",
         save="v8_fin_source_composition")

    _wire = float(OUT.loc[OUT.payment_rail.astype(str).str.upper().str.contains("WIRE", na=False),
                          "share_of_all_populated"].sum())
    _ach  = float(OUT.loc[OUT.payment_rail.astype(str).str.upper().str.contains("ACH", na=False),
                          "share_of_all_populated"].sum())
    kv([("outbound institution rows that are WIRE", round(_wire, 3)),
        ("outbound institution rows that are ACH", round(_ach, 3)),
        ("overall outbound has_fin", round(float((OUT.n_txn*OUT.has_fin).sum()/OUT.n_txn.sum()), 3)),
        ("verdict", "fin_out_n is substantially a WIRE signal" if _wire > 0.60 else
                    "institution coverage is spread across rails")],
       title="4c &middot; Reading 4b", save="v8_rail_verdict")


In [ ]:
# =====================================================================
# 4d-f · RAIL x INSTITUTION — BEHAVIOURAL, FROM DATA ON DISK
# =====================================================================
# The decisive half, and it costs nothing: v2's panel already carries
# amt_out_wire and fin_out_n per customer-month. If having a fin_out_n
# value is nearly the same event as sending a wire, the signal IS the rail.
feat = spark.read.parquet(v2("panel_pay_features")).filter(F.col("ym") >= DATE_START[:7])
RAILC = ["amt_out_ach", "amt_out_wire", "amt_out_check", "amt_out_card", "amt_out_rtp"]
have = [c for c in RAILC if c in feat.columns]

conf = feat.select("cust_pwr_id", "ym", "fin_out_n", "cpty_out_n", *have)
for c in have:
    conf = conf.withColumn("u_" + c.replace("amt_out_", ""),
                           (F.coalesce(F.col(c), F.lit(0.0)) > 0).cast("int"))
conf = conf.withColumn("has_fin", (F.coalesce(F.col("fin_out_n"), F.lit(0.0)) > 0).cast("int"))

X = (conf.groupBy("u_wire", "has_fin").agg(F.count(F.lit(1)).alias("customer_months"))
     .toPandas().pivot_table(index="u_wire", columns="has_fin",
                             values="customer_months", aggfunc="sum").fillna(0))
X.columns = [f"fin_out_n {'>0' if c else '=0'}" for c in X.columns]
X.index = ["no wire this month", "sent a wire"]
X["row share with fin"] = (X.iloc[:, -1] / X.sum(axis=1)).round(3)
disp(X.reset_index().rename(columns={"index": "wire usage"}),
     title="4d &middot; <b>Does a client have an institution count because it sent a wire?</b> "
           "Customer-months, whole panel", save="v8_wire_fin_crosstab")

# Conditional population: how often is fin present given each rail is used alone
rows = []
for c in have:
    tag = c.replace("amt_out_", "")
    only = conf.filter((F.col("u_" + tag) == 1) &
                       (sum([F.col("u_" + o.replace("amt_out_", "")) for o in have
                             if o != c]) == 0))
    n = only.count()
    rows.append(dict(rail_used_alone=tag, customer_months=n,
                     share_with_fin_out_n=(only.filter("has_fin=1").count()/n) if n else np.nan))
disp(pd.DataFrame(rows).sort_values("share_with_fin_out_n", ascending=False),
     title="4e &middot; Clients using <b>one rail only</b> in a month — do they get an institution "
           "count? This isolates the rail from the mix", save="v8_rail_only_fin")

# The consequence test: does fin_out_n predict WITHIN wire users?
RS = spark.read.parquet(v7("risk_set")) if exists(v7("risk_set")) else None
if RS is not None:
    _m = feat.select("cust_pwr_id", "m_idx",
                     (F.coalesce(F.col("amt_out_wire"), F.lit(0.0)) > 0).cast("int").alias("u_wire"))
    S = (RS.select("cust_pwr_id", "m_idx", "event_A", "ld_fin_out_n", "md_fin_out_n",
                   "ld_bal_live", "md_bal_live")
           .join(_m, ["cust_pwr_id", "m_idx"], "left"))
    SP = collect_pd(S.sample(False, 0.25, SEED), "risk set sample for the conditional test")
    SP["y"] = ((pd.to_numeric(SP.event_A, errors="coerce") > SP.m_idx) &
               (pd.to_numeric(SP.event_A, errors="coerce") <= SP.m_idx + PRIMARY_H)).astype(float)
    SP["s_fin"] = np.where(SP.md_fin_out_n == 0, -SP.ld_fin_out_n, np.nan)
    SP["s_bal"] = np.where(SP.md_bal_live  == 0, -SP.ld_bal_live,  np.nan)
    grp = []
    for lab, sel in [("all clients", SP.index),
                     ("wire users only", SP.index[SP.u_wire == 1]),
                     ("non-wire clients", SP.index[SP.u_wire.fillna(0) == 0])]:
        d = SP.loc[sel]
        grp.append(dict(population=lab, n=len(d), base_rate=float(d.y.mean()),
                        auc_fin_out_n=auc(d.y, d.s_fin), auc_bal_live=auc(d.y, d.s_bal),
                        share_scoreable_fin=float(np.isfinite(d.s_fin).mean())))
    disp(pd.DataFrame(grp).round(4),
         title="4f &middot; <b>Does the institution signal survive conditioning on wire use?</b> "
               "If its AUC among wire users is no better than among everyone, and it is "
               "unscoreable for non-wire clients, the signal is the rail",
         save="v8_fin_conditional_auc")
    note("RAIL", "Is the institution field a function of payment rail?",
         "see v8_rail_verdict and v8_fin_conditional_auc",
         "Structural half from the source table, behavioural half from panels already on disk. "
         "The consequence, if it is rail-determined: fin_out_n should be restated as a wire "
         "relationship count, and the inbound institution field — populated 86% — becomes the "
         "version worth building.")


## 5 · Counterparty key QA — before switching, not after

The plan is to re-key outbound counterparties from `unq_cpty_acct_id` (39.5% populated) to a
normalised `cpty_name` (99.9%). That is a 2.5× coverage gain and it is also a change of what a
"counterparty" *is*, so it gets measured first.

Two failure modes, opposite in direction and both silent:

- **Under-merge.** One entity spelled several ways becomes several counterparties. Counts inflate,
  churn becomes noise. Measured as distinct normalised names per counterparty account.
- **Over-merge.** Generic strings — `PAYROLL`, `TRANSFER`, `ACH CREDIT` — and processors collapse
  thousands of unrelated entities into one key. Counts deflate and every client looks like it pays
  the same partner. Measured as distinct accounts per normalised name, and controlled with a
  generic-name registry built on the **median-multiple** rule rather than a percentile, because a
  percentile flags a fixed fraction of names by construction.

Both are measurable on rows where the account id *and* the name are populated, which is 39.5% of
outbound and 92.7% of inbound — a large calibration set.

In [ ]:
# =====================================================================
# 5 · COUNTERPARTY KEY QA                                [OUTPUT BLOCK 4]
# =====================================================================
def norm_name(c):
    """Upper, strip punctuation, collapse whitespace, drop trailing legal
    suffixes. Deliberately conservative: this is exact-match-after-normalise,
    NOT fuzzy. Fuzzy across billions of rows is a separate project; the
    under-merge it leaves behind is quantified in 5a rather than assumed away."""
    x = F.upper(F.trim(F.col(c)))
    x = F.regexp_replace(x, r"[^A-Z0-9 ]", " ")
    x = F.regexp_replace(x, r"\s+", " ")
    for s in LEGAL_SUFFIX:
        x = F.regexp_replace(x, rf"(^| ){s}( |$)", " ")
    return F.trim(F.regexp_replace(x, r"\s+", " "))

if RUN_KEY_QA:
    t0 = time.time()
    both = (month_slice(AUDIT_MONTHS[1])
            .filter(F.col("direction").isin("outbound_cpty", "inbound_cpty"))
            .filter((_pop("unq_cpty_acct_id") == 1) & (_pop("cpty_name") == 1))
            .select("direction", "unq_cpty_acct_id", norm_name("cpty_name").alias("nkey"),
                    "cpty_fin_entity_name")
            .filter(F.length("nkey") >= 3).persist(StorageLevel.DISK_ONLY))

    under = (both.groupBy("direction", "unq_cpty_acct_id")
             .agg(F.countDistinct("nkey").alias("names_per_account")))
    over  = (both.groupBy("direction", "nkey")
             .agg(F.countDistinct("unq_cpty_acct_id").alias("accounts_per_name")))
    disp(under.groupBy("direction").agg(
            F.avg("names_per_account").alias("mean"),
            F.expr("percentile_approx(names_per_account, 0.5)").alias("p50"),
            F.expr("percentile_approx(names_per_account, 0.99)").alias("p99"),
            F.avg((F.col("names_per_account") > 1).cast("double")).alias("share_split")),
         title="5a &middot; <b>Under-merge.</b> Distinct normalised names per counterparty "
               "account. share_split is the fraction of real entities our name key would break "
               "into two or more", save="v8_undermerge")
    disp(over.groupBy("direction").agg(
            F.avg("accounts_per_name").alias("mean"),
            F.expr("percentile_approx(accounts_per_name, 0.5)").alias("p50"),
            F.expr("percentile_approx(accounts_per_name, 0.999)").alias("p999"),
            F.max("accounts_per_name").alias("max")),
         title="5b &middot; <b>Over-merge.</b> Distinct accounts behind one normalised name. The "
               "tail is what the generic registry has to remove", save="v8_overmerge")
    disp(over.filter("direction='outbound_cpty'").orderBy(F.desc("accounts_per_name")).limit(30),
         title="5c &middot; The worst offenders outbound — read these, they name the registry",
         n=30, save="v8_generic_candidates")

    # ── generic-name registry, median-multiple rule ────────────────────
    payers = (month_slice(AUDIT_MONTHS[1]).filter(F.col("direction") == "outbound_cpty")
              .filter(_pop("cpty_name") == 1)
              .select(norm_name("cpty_name").alias("nkey"), "mdm_id_pays")
              .filter(F.length("nkey") >= 3)
              .groupBy("nkey").agg(F.countDistinct("mdm_id_pays").alias("n_payers")))
    med = payers.approxQuantile("n_payers", [0.5], 0.001)[0] or 1.0
    thr = max(GENERIC_MEDIAN_MULT * med, GENERIC_MIN_PAYERS)
    GEN = payers.filter(F.col("n_payers") >= thr).persist(StorageLevel.DISK_ONLY)
    GEN.write.mode("overwrite").parquet(hp("generic_names"))
    kv([("median payers per counterparty name", med),
        ("generic threshold (median x %d, floored)" % GENERIC_MEDIAN_MULT, thr),
        ("names flagged generic", GEN.count()),
        ("share of outbound name-rows they carry",
         round(pct(GEN.join(payers, "nkey").agg(F.sum("n_payers")).collect()[0][0],
                   payers.agg(F.sum("n_payers")).collect()[0][0]), 4)),
        ("QA wall (s)", round(time.time()-t0))],
       title="5d &middot; Generic-name registry. These keys are <b>excluded from counterparty "
             "counts and churn</b> and tracked separately as processor exposure",
       save="v8_generic_registry")
    disp(GEN.orderBy(F.desc("n_payers")).limit(25),
         title="5e &middot; Top generic names — sanity-check that these are processors and "
               "payment-instruction strings, not real large counterparties", n=25)
    both.unpersist()
    note("KEY", "Is a normalised counterparty name a safe key?",
         "see v8_undermerge / v8_overmerge",
         "Under-merge inflates counterparty counts; over-merge deflates them. The generic registry "
         "handles the second. The first is left in and quantified — it is a level bias, and every "
         "feature here is read as a difference-in-differences against the client's own history, "
         "which absorbs a stable level bias.")


## 6 · Per-month enriched build

One Spark job per month, skipped if its output exists. A single unfiltered pass over the payment
table kills the SparkContext — that is a documented trap from v2 and the same pattern is used here.

Three artefacts per month, all restricted to the study population:

| Artefact | Grain | Feeds |
|---|---|---|
| `pairs/ym=` | customer × direction × key-type × key | counterparty and institution counts, churn, recurring-series breakage |
| `rail/ym=` | customer × direction × rail | rail-mix shift, and rail-conditional versions of the institution features |
| `selfpay/ym=` | customer × destination institution | self-payment to a named competitor |

Key types carried side by side rather than one replacing another: `cptyacct` (the v2 key),
`cptyname` (the new one), `fin` (institution). Keeping both counterparty keys is what makes the
re-keying decision measurable in §9 instead of assumed.

In [ ]:
# =====================================================================
# 6 · PER-MONTH BUILD — resumable                        [OUTPUT BLOCK 5]
# =====================================================================
# Account -> customer map. TRAP (02 §6): the naive join fans out to 241,499
# rows for 229,363 accounts and silently inflates ~5% of customers' volume.
# One row per account, latest link wins, assert.
dep = spark.table(DEP_TBL).filter(F.col("edw_tda_load_dt").between(DATE_START, DATE_END))
amap = (dep.select("acct_full_acct_id", "cust_pwr_id", "edw_tda_load_dt")
        .filter(F.col("cust_pwr_id").isNotNull() & F.col("acct_full_acct_id").isNotNull())
        .withColumn("rn", F.row_number().over(Window.partitionBy("acct_full_acct_id")
                                              .orderBy(F.desc("edw_tda_load_dt"))))
        .filter("rn = 1").select(F.col("acct_full_acct_id").alias("acct"), "cust_pwr_id")
        .persist(StorageLevel.DISK_ONLY))
_na, _nd = amap.count(), amap.select("acct").distinct().count()
assert _na == _nd, f"account map fanned out: {_na:,} rows for {_nd:,} accounts"
print(f"  account map: {_na:,} accounts (1:1 asserted)")

cust_names = (dep.select("cust_pwr_id", "cust_name").filter(F.col("cust_name").isNotNull())
              .groupBy("cust_pwr_id").agg(F.max("cust_name").alias("cust_name")))
GENERIC = spark.read.parquet(hp("generic_names")).select("nkey").withColumn("is_generic", F.lit(1))

YMS = pd.period_range(DATE_START[:7], DATE_END[:7], freq="M").strftime("%Y-%m").tolist()

def build_month(ym):
    lo = f"{ym}-01"; hi = (pd.Timestamp(lo)+pd.offsets.MonthEnd(1)).strftime("%Y-%m-%d")
    p = (spark.table(PAY_TBL).filter(F.col("trans_dt").between(lo, hi))
         .select("trans_dt", "trans_amt", "payment_rail", "category",
                 "mdm_id_pays", "mdm_id_receives", "pnc_dep_acct_pays", "pnc_dep_acct_receives",
                 "unq_cpty_acct_id", "cpty_name", "cpty_fin_entity_name"))
    out = (p.filter(F.col("mdm_id_pays").isNotNull() & F.col("mdm_id_receives").isNull())
           .withColumn("acct", F.col("pnc_dep_acct_pays")).withColumn("dir", F.lit("out")))
    inb = (p.filter(F.col("mdm_id_receives").isNotNull() & F.col("mdm_id_pays").isNull())
           .withColumn("acct", F.col("pnc_dep_acct_receives")).withColumn("dir", F.lit("in")))
    sides = (out.unionByName(inb).join(F.broadcast(amap), "acct", "inner")
             .withColumn("day", F.dayofmonth("trans_dt"))
             .withColumn("nkey", norm_name("cpty_name"))
             .withColumn("fkey", F.upper(F.trim(F.col("cpty_fin_entity_name"))))
             .withColumn("amt", F.col("trans_amt").cast("double")))

    # -- rail detail (also the per-customer input to the rail audit) ----
    (sides.groupBy("cust_pwr_id", "dir", "payment_rail").agg(
        F.count(F.lit(1)).alias("n_txn"), F.sum("amt").alias("amt"),
        F.sum(_pop("cpty_fin_entity_name")).alias("n_with_fin"),
        F.sum(_pop("unq_cpty_acct_id")).alias("n_with_acct"),
        F.sum(_pop("cpty_name")).alias("n_with_name"))
     .write.mode("overwrite").parquet(hp(f"rail/ym={ym}")))

    # -- pairs, three key types side by side ----------------------------
    def pairs_for(kt, keycol, src):
        d = src.filter(_pop(keycol) == 1).withColumn("k", F.col(keycol))
        if kt == "cptyname":
            d = (d.filter(F.length("k") >= 3).join(F.broadcast(GENERIC),
                          d["k"] == GENERIC["nkey"], "left")
                 .filter(F.col("is_generic").isNull()).drop("nkey", "is_generic"))
        return (d.groupBy("cust_pwr_id", "dir", "k").agg(
                    F.count(F.lit(1)).alias("n_txn"), F.sum("amt").alias("amt"),
                    F.countDistinct("day").alias("n_days"), F.avg("day").alias("day_mean"),
                    F.min("day").alias("day_min"), F.max("day").alias("day_max"))
                .withColumn("kt", F.lit(kt)))
    pr = (pairs_for("cptyacct", "unq_cpty_acct_id", sides)
          .unionByName(pairs_for("cptyname", "nkey", sides))
          .unionByName(pairs_for("fin", "fkey", sides)))
    # cap the tail: past PAIR_TOP_N by amount it is one-off payments, and it is
    # the join cost in §7, not the signal
    w = Window.partitionBy("cust_pwr_id", "dir", "kt").orderBy(F.desc("amt"), F.asc("k"))
    # ym is carried by the partition path only. Writing it as a column too
    # makes the partition-discovery read fail on a duplicate name.
    (pr.withColumn("rk", F.row_number().over(w)).filter(F.col("rk") <= PAIR_TOP_N)
       .write.mode("overwrite").parquet(hp(f"pairs/ym={ym}")))

    # -- self-payment to an external institution ------------------------
    sp = (sides.filter((F.col("dir") == "out") & (_pop("cpty_name") == 1))
          .join(F.broadcast(cust_names), "cust_pwr_id", "inner")
          # Spark has no jaro_winkler builtin (that is Impala/Snowflake). This is
          # deliberately conservative: exact normalised match, or a shared
          # 8-character prefix. It UNDER-detects self-payment, which biases the
          # feature toward zero rather than inventing one. Fuzzy matching over
          # ~500M rows a month is a separate exercise.
          .withColumn("cnorm", F.trim(F.regexp_replace(
              F.regexp_replace(F.upper(F.trim(F.col("cust_name"))), r"[^A-Z0-9 ]", " "),
              r"\s+", " ")))
          .withColumn("self_sim",
              F.when(F.col("nkey") == F.col("cnorm"), 1.0)
               .when((F.length("cnorm") >= 8) &
                     (F.substring(F.col("nkey"), 1, 8) == F.substring(F.col("cnorm"), 1, 8)), 0.92)
               .otherwise(0.0))
          .filter(F.col("self_sim") >= SELF_SIM_MIN))
    if NON_PNC_ONLY:
        sp = sp.filter(_pop("cpty_fin_entity_name") == 1)
    (sp.groupBy("cust_pwr_id", "fkey").agg(
        F.count(F.lit(1)).alias("n_txn"), F.sum("amt").alias("amt"),
        F.max("self_sim").alias("max_sim"))
     .write.mode("overwrite").parquet(hp(f"selfpay/ym={ym}")))

if RUN_BUILD:
    t0 = time.time()
    for i, ym in enumerate(YMS, 1):
        if exists(hp(f"pairs/ym={ym}")) and exists(hp(f"rail/ym={ym}")) \
           and exists(hp(f"selfpay/ym={ym}")):
            continue
        s = time.time(); build_month(ym)
        el = time.time()-t0
        print(f"  {ym} ({i}/{len(YMS)})  {time.time()-s:,.0f}s  "
              f"elapsed {el/60:,.1f}m  projected {el/i*len(YMS)/60:,.0f}m")
    print(f"  build complete in {(time.time()-t0)/60:,.1f}m")


## 7 · New feature families

All monthly, all keyed on `(cust_pwr_id, m_idx)` so they merge into the v7 risk set with a plain
join and nothing downstream changes.

| Family | Features | The claim |
|---|---|---|
| `cpty_name_out` | `cptyn_out_n`, `cptyn_lost`, `cptyn_retention`, `cptyn_new` | The counterparty count on a key populated 99.9% instead of 39.5% |
| `cpty_in` | `cpty_in_n`, `cptyi_lost`, `cptyi_retention` | "Their customers stop paying them" measured directly instead of through the 29%-covered PNC-to-PNC proxy |
| `fin_in` | `fin_in_n`, `fin_in_lost`, `fin_in_new` | The institution signal on the 86%-covered direction |
| `recurring` | `rec_active`, `rec_broken`, `rec_broken_share`, `rec_broken_amt_share` | A standing monthly payment that stopped — destroyed by monthly aggregation until now |
| `timing` | `day_mean_shift`, `day_dispersion`, `max_gap_days`, `days_since_top_cpty` | Paying later, or with less regularity, before paying less |
| `selfpay` | `selfpay_amt_share`, `selfpay_fin_n`, `selfpay_growth` | Money moving to the client's own name at another bank — the competitor construct |

`cptyacct` features are built alongside `cptyname` deliberately. §9 measures both, so re-keying is a
result rather than an assumption.

In [ ]:
# =====================================================================
# 7 · FEATURE ASSEMBLY                                   [OUTPUT BLOCK 6]
# =====================================================================
cust_month = spark.read.parquet(v2("panel_customer_month")).filter(F.col("ym") >= DATE_START[:7])
YMMAP = cust_month.select("ym", "m_idx").distinct().persist(StorageLevel.DISK_ONLY)
M_MIN, M_MAX = [int(x) for x in cust_month.agg(F.min("m_idx"), F.max("m_idx")).collect()[0]]
print(f"  m_idx {M_MIN}-{M_MAX} (ABSOLUTE index — every month constant is an offset from M_MIN)")

if RUN_FEATURES:
    t0 = time.time()
    PAIRS = (spark.read.option("basePath", hp("pairs")).parquet(hp("pairs"))
             .join(YMMAP, "ym", "inner").persist(StorageLevel.DISK_ONLY))

    # ── counterparty / institution counts, churn, retention ───────────
    wk = Window.partitionBy("cust_pwr_id", "dir", "kt", "k").orderBy("m_idx")
    seen = PAIRS.select("cust_pwr_id", "dir", "kt", "k", "m_idx", "amt", "n_txn",
                        "day_mean", "n_days")
    # a key is "lost" at t if it was in the reference window and is absent now.
    # Absence needs the full (customer, key) x month grid, so it is expressed as
    # a base-window count against a present-flag rather than an anti-join.
    grid = (seen.groupBy("cust_pwr_id", "dir", "kt", "k")
            .agg(F.min("m_idx").alias("first_m"), F.max("m_idx").alias("last_m")))
    # Range join against 31 months. Broadcast it — without the hint this plans
    # as a sort-merge over the pair panel and does not come back.
    months = F.broadcast(YMMAP.select("m_idx").distinct())
    full = (grid.join(months, F.col("m_idx").between(F.col("first_m"), F.col("last_m") + 6))
            .select("cust_pwr_id", "dir", "kt", "k", "m_idx")
            .join(seen.select("cust_pwr_id", "dir", "kt", "k", "m_idx",
                              F.lit(1).alias("present"), "amt"),
                  ["cust_pwr_id", "dir", "kt", "k", "m_idx"], "left")
            .withColumn("present", F.coalesce("present", F.lit(0)))
            .withColumn("amt", F.coalesce("amt", F.lit(0.0))))
    rw = wk.rangeBetween(-REC_WINDOW, -1)
    full = (full.withColumn("hits_prev", F.sum("present").over(rw))
                .withColumn("amt_prev",  F.avg("amt").over(rw)))

    cnt = (full.filter(F.col("present") == 1)
           .groupBy("cust_pwr_id", "dir", "kt", "m_idx")
           .agg(F.countDistinct("k").alias("n_k"),
                F.sum("amt").alias("amt_k"),
                F.sum((F.col("hits_prev") == 0).cast("int")).alias("new_k")))
    lost = (full.filter((F.col("present") == 0) & (F.col("hits_prev") >= 1))
            .groupBy("cust_pwr_id", "dir", "kt", "m_idx")
            .agg(F.count(F.lit(1)).alias("lost_k"), F.sum("amt_prev").alias("lost_amt")))
    # recurring series: standing (>= REC_MIN_HITS of REC_WINDOW) and now absent
    rec = (full.withColumn("standing", (F.col("hits_prev") >= REC_MIN_HITS).cast("int"))
           .groupBy("cust_pwr_id", "dir", "kt", "m_idx")
           .agg(F.sum("standing").alias("rec_active"),
                # F.sum(<boolean>) raises "sum requires numeric" — every count
                # here goes through F.when(...).otherwise(0)
                F.sum(F.when((F.col("standing") == 1) & (F.col("present") == 0), 1)
                       .otherwise(0)).alias("rec_broken"),
                F.sum(F.when((F.col("standing") == 1) & (F.col("present") == 0),
                             F.col("amt_prev")).otherwise(0.0)).alias("rec_broken_amt"),
                F.sum(F.when(F.col("standing") == 1, F.col("amt_prev"))
                       .otherwise(0.0)).alias("rec_active_amt")))

    def wide(df, cols, prefix_map):
        out = None
        for (d, kt), pref in prefix_map.items():
            sl = df.filter((F.col("dir") == d) & (F.col("kt") == kt)).drop("dir", "kt")
            sl = sl.select("cust_pwr_id", "m_idx",
                           *[F.col(c).alias(f"{pref}_{c}") for c in cols if c in sl.columns])
            out = sl if out is None else out.join(sl, ["cust_pwr_id", "m_idx"], "outer")
        return out

    PM = {("out", "cptyname"): "cptyn_out", ("out", "cptyacct"): "cptya_out",
          ("out", "fin"): "fin_out2",     ("in", "cptyname"): "cptyn_in",
          ("in", "fin"): "fin_in"}
    FEAT_CNT = wide(cnt, ["n_k", "amt_k", "new_k"], PM)
    FEAT_LST = wide(lost, ["lost_k", "lost_amt"], PM)
    FEAT_REC = wide(rec, ["rec_active", "rec_broken", "rec_broken_amt", "rec_active_amt"], PM)
    NEWF = (FEAT_CNT.join(FEAT_LST, ["cust_pwr_id", "m_idx"], "outer")
                    .join(FEAT_REC, ["cust_pwr_id", "m_idx"], "outer"))
    for p in set(PM.values()):
        if f"{p}_lost_k" in NEWF.columns and f"{p}_n_k" in NEWF.columns:
            NEWF = NEWF.withColumn(f"{p}_retention",
                F.when((F.coalesce(F.col(f"{p}_n_k"), F.lit(0.0)) +
                        F.coalesce(F.col(f"{p}_lost_k"), F.lit(0.0))) > 0,
                       F.col(f"{p}_n_k") /
                       (F.col(f"{p}_n_k") + F.col(f"{p}_lost_k"))))
        if f"{p}_rec_broken" in NEWF.columns:
            NEWF = NEWF.withColumn(f"{p}_rec_broken_share",
                F.when(F.col(f"{p}_rec_active") > 0,
                       F.col(f"{p}_rec_broken")/F.col(f"{p}_rec_active")))
            NEWF = NEWF.withColumn(f"{p}_rec_broken_amt_share",
                F.when(F.col(f"{p}_rec_active_amt") > 0,
                       F.col(f"{p}_rec_broken_amt")/F.col(f"{p}_rec_active_amt")))

    # ── timing ────────────────────────────────────────────────────────
    tim = (PAIRS.filter((F.col("dir") == "out") & (F.col("kt") == "cptyname"))
           .groupBy("cust_pwr_id", "m_idx").agg(
               F.avg("day_mean").alias("tim_day_mean"),
               F.stddev("day_mean").alias("tim_day_sd"),
               F.avg("n_days").alias("tim_days_active"),
               F.max("day_max").alias("tim_day_last")))
    wt = Window.partitionBy("cust_pwr_id").orderBy("m_idx").rangeBetween(CHG_LAG_FAR, CHG_LAG_NEAR)
    tim = (tim.withColumn("tim_day_base", F.avg("tim_day_mean").over(wt))
              .withColumn("tim_day_shift", F.col("tim_day_mean") - F.col("tim_day_base"))
              .drop("tim_day_base"))

    # ── rail mix shift ────────────────────────────────────────────────
    RAILM = (spark.read.option("basePath", hp("rail")).parquet(hp("rail"))
             .join(YMMAP, "ym", "inner").filter(F.col("dir") == "out"))
    tot = RAILM.groupBy("cust_pwr_id", "m_idx").agg(F.sum("amt").alias("tot_amt"),
                                                    F.sum("n_txn").alias("tot_txn"))
    mix = (RAILM.join(tot, ["cust_pwr_id", "m_idx"])
           .withColumn("share", F.when(F.col("tot_amt") > 0, F.col("amt")/F.col("tot_amt")))
           .select("cust_pwr_id", "m_idx", "payment_rail", "share"))
    wm = (Window.partitionBy("cust_pwr_id", "payment_rail").orderBy("m_idx")
          .rangeBetween(CHG_LAG_FAR, CHG_LAG_NEAR))
    mix = mix.withColumn("share_base", F.avg("share").over(wm))
    RAILSHIFT = (mix.groupBy("cust_pwr_id", "m_idx")
                 .agg((F.sum(F.abs(F.coalesce(F.col("share"), F.lit(0.0)) -
                                   F.coalesce(F.col("share_base"), F.lit(0.0))))/2)
                      .alias("railmix_shift"),
                      F.sum(F.when(F.col("share_base") > 0.05,
                            (F.coalesce(F.col("share"), F.lit(0.0)) < 0.2*F.col("share_base"))
                            .cast("int")).otherwise(0)).alias("railmix_rails_collapsed")))

    # ── concentration ─────────────────────────────────────────────────
    wc = Window.partitionBy("cust_pwr_id", "m_idx").orderBy(F.desc("amt"))
    conc = (PAIRS.filter((F.col("dir") == "out") & (F.col("kt") == "cptyname"))
            .withColumn("rk", F.row_number().over(wc))
            .groupBy("cust_pwr_id", "m_idx").agg(
                F.sum("amt").alias("_t"),
                F.sum(F.when(F.col("rk") == 1, F.col("amt")).otherwise(0.0)).alias("_t1"),
                F.sum(F.when(F.col("rk") <= 3, F.col("amt")).otherwise(0.0)).alias("_t3"))
            .withColumn("conc_top1", F.when(F.col("_t") > 0, F.col("_t1")/F.col("_t")))
            .withColumn("conc_top3", F.when(F.col("_t") > 0, F.col("_t3")/F.col("_t")))
            .drop("_t", "_t1", "_t3"))

    # ── self-payment ──────────────────────────────────────────────────
    SP = (spark.read.option("basePath", hp("selfpay")).parquet(hp("selfpay"))
          .join(YMMAP, "ym", "inner")
          .groupBy("cust_pwr_id", "m_idx").agg(
              F.sum("amt").alias("selfpay_amt"), F.countDistinct("fkey").alias("selfpay_fin_n"),
              F.sum("n_txn").alias("selfpay_txn")))

    # ── account trajectory & dormancy ─────────────────────────────────
    wa = Window.partitionBy("cust_pwr_id").orderBy("m_idx")
    ACC = (cust_month.select("cust_pwr_id", "m_idx", "n_accts", "n_accts_live")
           .withColumn("acc_live_base", F.avg("n_accts_live").over(wa.rangeBetween(CHG_LAG_FAR, CHG_LAG_NEAR)))
           .withColumn("acc_live_ratio", F.when(F.col("acc_live_base") > 0,
                       F.col("n_accts_live")/F.col("acc_live_base")))
           .withColumn("acc_closed_share", F.when(F.col("n_accts") > 0,
                       1.0 - F.col("n_accts_live")/F.col("n_accts")))
           .select("cust_pwr_id", "m_idx", "acc_live_ratio", "acc_closed_share"))

    V8F = (NEWF.join(tim, ["cust_pwr_id", "m_idx"], "outer")
                .join(RAILSHIFT, ["cust_pwr_id", "m_idx"], "outer")
                .join(conc, ["cust_pwr_id", "m_idx"], "outer")
                .join(SP, ["cust_pwr_id", "m_idx"], "outer")
                .join(ACC, ["cust_pwr_id", "m_idx"], "outer"))
    V8F.write.mode("overwrite").partitionBy("m_idx").parquet(hp("features_v8"))
    V8F = spark.read.parquet(hp("features_v8")).persist(StorageLevel.DISK_ONLY)

    NEWCOLS = [c for c in V8F.columns if c not in ("cust_pwr_id", "m_idx")]
    covrows = (V8F.filter(F.col("m_idx") >= M_MIN + DD_WARMUP_M)
               .agg(F.count(F.lit(1)).alias("rows"),
                    *[F.avg(F.col(c).isNotNull().cast("double")).alias(c) for c in NEWCOLS])
               .toPandas().T)
    covrows.columns = ["value"]
    disp(covrows.reset_index().rename(columns={"index": "feature"}),
         title="7a &middot; <b>Coverage of every new feature.</b> Compare "
               "<code>cptyn_out_n_k</code> against <code>cptya_out_n_k</code> and "
               "<code>fin_in_n_k</code> against the old 42% — this is the whole v8 thesis in one "
               "table", n=80, save="v8_new_feature_coverage")
    kv([("new features built", len(NEWCOLS)),
        ("customer-months", V8F.count()),
        ("feature build wall (min)", round((time.time()-t0)/60, 1))],
       title="7b &middot; Build", save="v8_feature_build")


## 8 · dd, risk set, and the merged panel

Same difference-in-differences construction as v6/v7 — `value_t / mean(t−6…t−4)`, then divided by
the frozen peer group's median for the same calendar month — applied to the new features so they
are on the same scale as the old ones and `median_dd = 1.000` remains the construction check.

Trend and run-length are added here for **every** feature, old and new: slope over the trailing
three months and consecutive months below peer. v7 read each signal as a single month's level,
which cannot distinguish a client drifting down for five months from one that dipped once.

In [ ]:
# =====================================================================
# 8 · dd + TREND + MERGE INTO THE v7 RISK SET            [OUTPUT BLOCK 7]
# =====================================================================
if "V8F" not in dir():          # §7 may have been skipped or run in an earlier session
    V8F = spark.read.parquet(hp("features_v8")).persist(StorageLevel.DISK_ONLY)
anchor = read_first([v6("peer_anchor"), v7("peer_anchor")], "peer_anchor")
assert anchor is not None, "peer anchor missing — run v6/v7 §2 first"
RS7 = spark.read.parquet(v7("risk_set"))
assert RS7 is not None
NEWCOLS = [c for c in V8F.columns if c not in ("cust_pwr_id", "m_idx")]

_stack = ", ".join([f"'{f}', CAST(`{f}` AS DOUBLE)" for f in NEWCOLS])
lg = (V8F.join(anchor.select("cust_pwr_id", "peer_key"), "cust_pwr_id", "left")
      .join(YMMAP, "m_idx", "left")
      .select("cust_pwr_id", "m_idx", "ym", "peer_key",
              F.expr(f"stack({len(NEWCOLS)}, {_stack}) as (feature, value)")))
wr = (Window.partitionBy("cust_pwr_id", "feature").orderBy("m_idx")
      .rangeBetween(CHG_LAG_FAR, CHG_LAG_NEAR))   # rangeBetween, never rowsBetween
lg = (lg.withColumn("ref", F.avg("value").over(wr))
        .withColumn("nref", F.count("value").over(wr))
        .withColumn("chg", F.when((F.col("nref") >= CHG_MIN_OBS) & (F.col("ref") > MIN_REF),
                                  F.col("value")/F.col("ref"))))
pc = (lg.groupBy("ym", "peer_key", "feature")
      .agg(F.count("chg").alias("pn"), F.expr("percentile_approx(chg, 0.5)").alias("pchg"))
      .filter((F.col("pn") >= PEER_MIN_N) & (F.abs(F.col("pchg")) > 1e-6)))
lg = (lg.join(pc, ["ym", "peer_key", "feature"], "left")
        .withColumn("dd", F.when(F.col("chg").isNotNull() & F.col("pchg").isNotNull(),
                                 F.col("chg")/F.col("pchg"))))
disp(lg.groupBy("feature").agg(
        F.avg(F.col("dd").isNotNull().cast("double")).alias("share_with_dd"),
        F.expr("percentile_approx(dd, 0.5)").alias("median_dd")).orderBy("feature"),
     title="8a &middot; dd on the new features. <b>median_dd at 1.000 is the construction check</b>",
     n=80, save="v8_new_dd_coverage")

wide8 = (lg.groupBy("cust_pwr_id", "m_idx").pivot("feature", NEWCOLS)
         .agg(F.first("dd", True).alias("dd"), F.first("value", True).alias("v")))
sel = [F.col("cust_pwr_id"), F.col("m_idx")]
for f in NEWCOLS:
    dd = F.col(f"{f}_dd")
    sel.append(F.when(dd.isNotNull(), F.log(F.greatest(F.least(dd, F.lit(DD_CLIP[1])),
                                                       F.lit(DD_CLIP[0])))).alias(f"ld_{f}"))
    sel.append(F.when(dd.isNull(), F.lit(1.0)).otherwise(F.lit(0.0)).alias(f"md_{f}"))
    sel.append(F.col(f"{f}_v").alias(f"raw_{f}"))
wide8 = wide8.select(*sel)

# ── trend and run-length, for OLD and NEW features alike ──────────────
OLD_LD = [c for c in RS7.columns if c.startswith("ld_")]
trend = RS7.select("cust_pwr_id", "m_idx", *OLD_LD).join(
            wide8.select("cust_pwr_id", "m_idx", *[f"ld_{f}" for f in NEWCOLS]),
            ["cust_pwr_id", "m_idx"], "outer")
wt3 = Window.partitionBy("cust_pwr_id").orderBy("m_idx").rangeBetween(-2, 0)
for c in OLD_LD + [f"ld_{f}" for f in NEWCOLS]:
    trend = trend.withColumn("tr_" + c[3:],
        F.col(c) - F.avg(F.col(c)).over(wt3))          # this month vs its own 3m mean
trend = trend.select("cust_pwr_id", "m_idx", *[c for c in trend.columns if c.startswith("tr_")])

RISK8 = (RS7.join(wide8, ["cust_pwr_id", "m_idx"], "left")
            .join(trend, ["cust_pwr_id", "m_idx"], "left"))
RISK8.write.mode("overwrite").partitionBy("m_idx").parquet(hp("risk_set_v8"))
RISK8 = spark.read.parquet(hp("risk_set_v8")).persist(StorageLevel.DISK_ONLY)
kv([("risk-set columns, v7", len(RS7.columns)),
    ("risk-set columns, v8", len(RISK8.columns)),
    ("new dd features", len(NEWCOLS)),
    ("trend features", len(OLD_LD) + len(NEWCOLS))],
   title="8b &middot; The merged panel", save="v8_panel_shape")


## 9 · Ablation — every block measured separately

Same clients, same seven rolling origins, same horizon as v7, so `M4_pay_plus_bal` at **AUC 0.795 /
precision 0.651 at K=250** is the fixed baseline every row is compared against. Blocks are added
one at a time and then all together, and the two counterparty keys are compared head to head so the
re-keying decision is a measurement.

A block that does not move AUC **and** does not move precision at the operating capacity does not
ship, however good the argument for it was.

In [ ]:
# =====================================================================
# 9 · ABLATION                                           [OUTPUT BLOCK 8]
# =====================================================================
if RUN_ABLATION:
    ORIGINS = list(range(M_MIN + ORIGIN_START_OFF, M_MAX - PRIMARY_H + 1))
    assert 1 <= len(ORIGINS) <= MAX_ORIGINS, (
        f"{len(ORIGINS)} origins — m_idx is ABSOLUTE ({M_MIN}-{M_MAX}) and "
        f"ORIGIN_START_OFF is an OFFSET")
    print(f"  origins m_idx {ORIGINS[0]}-{ORIGINS[-1]} ({len(ORIGINS)} folds)")

    ALLC = RISK8.columns
    def block(pred): return sorted([c for c in ALLC if pred(c)])
    V7_LD  = block(lambda c: c.startswith("ld_") and not any(
                    c.startswith("ld_"+p) for p in ("cptyn_", "cptya_", "fin_out2", "cptyn_in",
                                                    "fin_in", "tim_", "railmix", "conc_",
                                                    "selfpay", "acc_")))
    V7_MD  = ["md_" + c[3:] for c in V7_LD if "md_" + c[3:] in ALLC]
    def blk(pfx):
        ld = block(lambda c: c.startswith("ld_" + pfx))
        return ld + [("md_" + c[3:]) for c in ld if ("md_" + c[3:]) in ALLC]
    BLOCKS = {
      "trend":          block(lambda c: c.startswith("tr_")),
      "railmix":        blk("railmix"),
      "concentration":  blk("conc_"),
      "accounts":       blk("acc_"),
      "cpty_name_out":  blk("cptyn_out"),
      "cpty_acct_out":  blk("cptya_out"),
      "cpty_in":        blk("cptyn_in"),
      "fin_in":         blk("fin_in"),
      "recurring":      [c for c in ALLC if "rec_" in c and c.startswith(("ld_", "md_"))],
      "timing":         blk("tim_"),
      "selfpay":        blk("selfpay"),
    }
    BLOCKS = {k: v for k, v in BLOCKS.items() if v}
    disp(pd.DataFrame([dict(block=k, n_features=len(v)) for k, v in BLOCKS.items()]),
         title="9a &middot; Blocks entering the ablation", save="v8_blocks")

    NEED = sorted(set(["cust_pwr_id", "m_idx", "event_A", "event_B", "last_m"] +
                      V7_LD + V7_MD + [c for v in BLOCKS.values() for c in v]))
    NEED = [c for c in NEED if c in ALLC]
    is_pos = (F.col("event_A").between(F.col("m_idx")+1, F.col("m_idx")+max(HORIZONS)) |
              F.col("event_B").between(F.col("m_idx")+1, F.col("m_idx")+max(HORIZONS)))
    TR = collect_pd(RISK8.filter(F.col("m_idx") <= max(ORIGINS) - min(HORIZONS))
                    .withColumn("_u", (F.abs(F.hash(F.concat_ws("|", "cust_pwr_id",
                                F.col("m_idx").cast("string"), F.lit(SEED)))) % 100000)/100000.0)
                    .filter(is_pos | (F.col("_u") < NEG_SAMPLE)).select(*NEED), "TRAIN")
    TE = {t: collect_pd(RISK8.filter(F.col("m_idx") == t).select(*NEED), f"TEST {t}")
          for t in ORIGINS}

    def prep(d):
        d = d.copy()
        for c in d.columns:
            if c.startswith("ld_") or c.startswith("tr_"):
                d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0.0)
            elif c.startswith("md_"):
                d[c] = pd.to_numeric(d[c], errors="coerce").fillna(1.0)
        return d
    def label(d, H, defn=PRIMARY_DEF):
        ev = pd.to_numeric(d["event_A" if defn == "A_full_exit" else "event_B"], errors="coerce")
        t = pd.to_numeric(d["m_idx"], errors="coerce")
        y = ((ev > t) & (ev <= t + H)).astype(float)
        obs = (t + H <= M_MAX) | (y == 1)
        o = d.loc[obs].copy(); o["y"] = y.loc[obs].values
        return o
    TRp = prep(TR); TEp = {k: prep(v) for k, v in TE.items()}

    SPECS = {"v7_baseline": V7_LD + V7_MD}
    for k, v in BLOCKS.items():
        SPECS[f"+ {k}"] = V7_LD + V7_MD + v
    SPECS["+ everything"] = V7_LD + V7_MD + sorted({c for v in BLOCKS.values() for c in v})

    rows, tk = [], []
    for Tm in ORIGINS:
        tr = label(TRp[TRp.m_idx <= Tm - PRIMARY_H], PRIMARY_H)
        te = label(TEp[Tm], PRIMARY_H)
        if tr.y.sum() < MIN_TRAIN_POS or te.y.sum() < 1: continue
        pb = float(te.y.mean())
        for nm, cols in SPECS.items():
            cols = [c for c in cols if c in tr.columns]
            sp = fit_spec(tr, cols); s = apply_spec(sp, te)
            rows.append(dict(spec=nm, origin=Tm, auc=auc(te.y.values, s), base_rate=pb,
                             n_feat=len(sp["cols"]) if sp else 0))
            m = topk_metrics(s, te.y.values, CAPACITY, p_base=pb)
            m.insert(0, "spec", nm); tk.append(m)
    AB = pd.DataFrame(rows); TK = pd.concat(tk, ignore_index=True)
    AB.to_csv(OUT_DIR / "v8_ablation_folds.csv", index=False)

    P = TK.groupby(["spec", "k"], as_index=False).agg(tp=("tp", "sum"), alerts=("k", "sum"))
    P["precision"] = P.tp/P.alerts
    SUM = (AB.groupby("spec", as_index=False).agg(auc=("auc", "mean"), sd=("auc", "std"),
                                                  n_feat=("n_feat", "max"))
           .merge(P[P.k == QUEUE_K][["spec", "precision"]], on="spec", how="left"))
    b_auc = float(SUM.loc[SUM.spec == "v7_baseline", "auc"].iloc[0])
    b_prc = float(SUM.loc[SUM.spec == "v7_baseline", "precision"].iloc[0])
    SUM["d_auc"] = (SUM.auc - b_auc).round(4)
    SUM["d_precision"] = (SUM.precision - b_prc).round(4)
    SUM["ships"] = np.where((SUM.d_auc > 0.004) | (SUM.d_precision > 0.010), "yes",
                    np.where(SUM.spec == "v7_baseline", "&mdash;", "no"))
    disp(SUM.sort_values("d_auc", ascending=False).round(4),
         title=f"9b &middot; <b>Ablation.</b> Baseline is v7 at AUC {b_auc:.3f} / precision "
               f"{b_prc:.3f} at K={QUEUE_K}. A block that moves neither does not ship",
         n=30, save="v8_ablation")
    _kn = SUM[SUM.spec == "+ cpty_name_out"]; _ka = SUM[SUM.spec == "+ cpty_acct_out"]
    kv([("baseline AUC (v7)", round(b_auc, 4)),
        ("best single block", str(SUM[SUM.spec != "v7_baseline"].sort_values("d_auc").iloc[-1].spec)),
        ("everything together, AUC", round(float(SUM.loc[SUM.spec == "+ everything", "auc"].iloc[0]), 4)),
        ("everything together, precision at K", round(float(SUM.loc[SUM.spec == "+ everything", "precision"].iloc[0]), 4)),
        ("name-keyed counterparties, d_auc", float(_kn.d_auc.iloc[0]) if len(_kn) else None),
        ("account-keyed counterparties, d_auc", float(_ka.d_auc.iloc[0]) if len(_ka) else None),
        ("re-keying verdict", ("name key wins" if len(_kn) and len(_ka) and
                               float(_kn.d_auc.iloc[0]) > float(_ka.d_auc.iloc[0]) else "inconclusive"))],
       title="9c &middot; Verdict", save="v8_ablation_verdict")
    note("ABLATE", "Which new feature families actually earn their place?",
         f"best block {SUM[SUM.spec!='v7_baseline'].sort_values('d_auc').iloc[-1].spec}",
         "Same clients, same folds, same horizon as v7, so every delta is against a fixed "
         "baseline. Customer attributes are deliberately absent — they arrive next and their lift "
         "should be measured against this, not against v7.")


---

## After this run

1. **§4 first.** If the institution field is rail-determined, `fin_out_n` gets restated in every
   document as a wire-relationship count, and `fin_in_n` — 86% covered — becomes the version worth
   briefing. If it is not, the coverage story is about the account-id key alone and §5's re-keying
   carries the whole gain.

2. **§9 decides what ships.** Anything with `ships = no` gets deleted, not parked. A feature that
   moved nothing on seven folds against a fixed baseline will not start working later, and carrying
   it costs maintenance and review surface.

3. **Then, and only then, add customer attributes.** Segment, NAICS, size band, product holdings.
   Their lift is measured against `+ everything` from §9, so what the attributes add is separable
   from what the payment rebuild added. Doing it the other way round makes both unmeasurable.

4. **Two things this notebook still does not do.** Fuzzy counterparty matching — the name key is
   exact-match-after-normalise, and §5a quantifies what that leaves on the table. And intra-month
   recurrence: a series is detected from monthly presence, so a twice-monthly payment that drops to
   monthly is invisible. Both are refinements on a working construct rather than gaps in it.
